# ATAC import: cell-state accessibility tracks

Imports the 10x multiome ATAC of the gastrulation chimeras and writes per-genotype, per-cell-state misha tracks. No figure panels.

- Genotypes: Dnmt-TKO, Dnmt-DKO and Tet-TKO (KO and host fractions), WT (`WT_lg`, `WT_sm`).
- Cells are grouped by the RNA cell type of their metacell; reads are pooled into 20 bp tracks `mut_multi2.<genotype>.<cell state>`, plus a `.marginal` track over all cells.
- Used by `prep/atac_model.ipynb` and the gastrulation ATAC notebooks (`Fig3`, `Fig4`, `Fig5`, `ExtData_ATAC`).
- Functions: `scripts/ATAC_import.r` (built on `mcATAC`).

Caveats:
- This is a working log, not a clean pipeline. Only the setup cells and three cells of the first section were executed in the stored copy.
- The paper's peak set and coverage model come from `prep/atac_model.ipynb`.

In [ ]:
source("../paths.R")

In [1]:
library(tidyverse)
library(tgutil)
library(mcATAC)
library(anndata)
library(Matrix)
library(cli)
library(glue)
library(readr)
library(plyr)
library(ggpubr)
library(misha.ext)

In [6]:
setwd(embdnmt_dir)

In [3]:
gsetroot(mm10_trackdb)

In [4]:
source(file.path(repo_dir, "scripts/ATAC_import.r"))

## Task flags

- `reimport_atac`: re-import the 10x outputs (`import_all_from_10x`).
- `rewrite_MCCs`: rebuild the cell-group counts and save `raw_data`; otherwise load the cached `raw_data.RDS` from `multiome_dir`.
- `recompute_tracks`: rewrite the misha tracks of all eight genotypes.
- `reinit = T` with `reimport_atac = T` re-imports all the 10x data.

In [ ]:
# Task flags
reinit = T

reimport_atac = T
rewrite_MCCs = F
recompute_tracks = F

## Generate cell-state tracks

The import, step by step:
- `read_SCCs`: per-cell ATAC counts from `tg_atac_dir/mm_gastru/<genotype>/outs/reads/`.
- `organize_md`: maps each cell to its metacell and atlas cell type (`./data/MC_models/multiome_<model>_*.h5ad`, `./data/ref/cell_md.csv`, `./data/ref/cell_type_key.csv`).
- `write_MCCs`: pools cells per genotype and cell type; drops `Missing`, `Excluded`, `Outliers`.
- Saves `raw_data` to `data/output/raw_data.RDS` (via `out_path`).

In [ ]:
raw_data <- list()
raw_data <- read_SCCs(raw_data)

In [ ]:
raw_data <- organize_md(raw_data)

In [ ]:
raw_data2 <- write_MCCs(raw_data)

In [ ]:
write_rds(raw_data2, file = out_path("./data/output/raw_data.RDS"))

In [ ]:
gdb.reload()

Writes the tracks of `TET_TKO_host`, `WT_sm` and `WT_lg` (the executed cells in the stored copy).
- `write_tracks` writes 20 bp, unnormalized tracks plus a marginal track, then drops the `mc` prefix from the track names.
- It overwrites `mut_multi2.<genotype>.*` in the shared misha database, with no flag guarding it.
- It leaves the working directory inside the track database (`setwd`).

In [5]:
raw_data <- read_rds(file = paste0(multiome_dir, "/tidy_data/raw_data.RDS"))

In [10]:
raw_data <- list()
raw_data <- read_SCCs(raw_data)
raw_data <- organize_md(raw_data)
raw_data <- write_MCCs(raw_data)

In [11]:
write_tracks(raw_data, c("TET_TKO_host", "WT_sm", "WT_lg"))

The same import as one cell, controlled by the task flags.

In [ ]:
### Obtain "raw_data" ###
if (reinit) {
    
    # Import ATAC raw data, write SCC objoects at base dir
    if (reimport_atac){
        import_all_from_10x()  
    }
    
    if (rewrite_MCCs){
        
        raw_data <- list()
        raw_data <- read_SCCs(raw_data)
        raw_data <- organize_md(raw_data)
        raw_data <- write_MCCs(raw_data)
        write_rds(raw_data, file = out_path("./data/output/raw_data.RDS"))
    
    } else {
        raw_data <- read_rds(file = paste0(multiome_dir, "/tidy_data/raw_data.RDS"))
    }
    
    # Write track per cell type, normalize each track
    if (recompute_tracks){
        write_tracks(raw_data)
    }
    
    # # 
    
}